# Guardrails Testing
## Validating SQL safety rules against malicious queries

In [ ]:
import json
from pathlib import Path
import pandas as pd
from guardrails.sql_validator import validator
from guardrails.parser import parse_sql, extract_dml_type
from guardrails.rules import DEFAULT_RULES

In [ ]:
# Load test data
with open(Path("../tests/test_data/malicious_queries.json")) as f:
    malicious = json.load(f)

print(f"Loaded {len(malicious)} test cases")
pd.DataFrame(malicious)

In [ ]:
# Test each malicious query
results = []
for item in malicious:
    result = validator.validate(item["sql"])
    expected_blocked = item["expected_block"]
    actually_blocked = not result.valid
    
    results.append({
        "id": item["id"],
        "sql": item["sql"][:60],
        "category": item["category"],
        "expected_block": expected_blocked,
        "blocked": actually_blocked,
        "pass": expected_blocked == actually_blocked,
        "errors": [str(e) for e in result.errors],
        "warnings": result.warnings,
    })

df = pd.DataFrame(results)
passed = df["pass"].sum()
print(f"Passed: {passed}/{len(df)}")
print()
df

In [ ]:
# Show failures if any
failures = df[~df["pass"]]
if len(failures):
    print("FAILURES:")
    display(failures)
else:
    print("All test cases passed!")

In [ ]:
# Analyze which rules catch what
print("Active rules:")
for rule in DEFAULT_RULES:
    print(f"  {rule.rule_id}: {rule.description} ({rule.action})")

In [ ]:
# Test edge cases
edge_cases = [
    "SELECT '--' as comment_test",  # comment in string literal - should pass
    "SELECT * FROM products WHERE name LIKE '%--%'",  # LIKE with dashes
    "With cte AS (SELECT * FROM products) SELECT * FROM cte LIMIT 5",  # CTE
]

for sql in edge_cases:
    result = validator.validate(sql)
    print(f"SQL: {sql[:60]}...")
    print(f"  Valid: {result.valid}, Errors: {result.errors}, Warnings: {result.warnings}")
    print()